# Seminar 2. Custom PyTorch Operators

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [1]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

CombinedModel(
  (block1): LinearReLUBlock(
    (linear): Linear(in_features=4, out_features=8, bias=True)
    (activation): ReLU()
  )
  (block2): LinearTanhBlock(
    (linear): Linear(in_features=8, out_features=8, bias=True)
    (activation): Tanh()
  )
  (output): Linear(in_features=8, out_features=2, bias=True)
)


# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

In [2]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


Registered parameters:
block1.linear.weight torch.Size([8, 4])
block1.linear.bias torch.Size([8])
block2.linear.weight torch.Size([8, 8])
block2.linear.bias torch.Size([8])
output.weight torch.Size([2, 8])
output.bias torch.Size([2])


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

In [3]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad


Gradient computed for output layer: True


tensor([[-0.0618, -0.0368, -0.0450, -0.0805, -0.0512,  0.0988, -0.0830,  0.0118],
        [ 0.3528,  0.0355,  0.6796, -0.0623,  0.3421, -0.5362,  0.7406, -0.2053]])

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

In [4]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

Model device: cpu
Model dtype: torch.float32


### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

In [5]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.



In [6]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


Training mode output:
 tensor([[-0.5771,  0.3109],
        [-0.2447, -0.2427],
        [-0.3054,  0.1277],
        [-0.0535,  0.4059],
        [-0.7095,  0.5248]], grad_fn=<AddmmBackward0>)
Evaluation mode output:
 tensor([[-0.4357, -0.1346],
        [-0.2003, -0.0145],
        [-0.1299, -0.1811],
        [-0.4276, -0.1327],
        [ 0.1917,  0.0478]])


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.



In [7]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------
    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)


Output no_grad method:
 tensor([[-0.3564,  0.1641],
        [-1.3147,  0.1616],
        [-1.4203, -0.7584],
        [-0.4611, -0.3633],
        [-0.4615, -0.3144]])
Output inference_mode method:
 tensor([[-0.3564,  0.1641],
        [-1.3147,  0.1616],
        [-1.4203, -0.7584],
        [-0.4611, -0.3633],
        [-0.4615, -0.3144]])
Output no_grad context manager:
 tensor([[-0.3564,  0.1641],
        [-1.3147,  0.1616],
        [-1.4203, -0.7584],
        [-0.4611, -0.3633],
        [-0.4615, -0.3144]])
Output inference_mode context manager:
 tensor([[-0.3564,  0.1641],
        [-1.3147,  0.1616],
        [-1.4203, -0.7584],
        [-0.4611, -0.3633],
        [-0.4615, -0.3144]])


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [8]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

fc.weight: requires_grad=False
fc.bias: requires_grad=False
Output shape: torch.Size([5, 2])


# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [9]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self

class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)


CustomBlock set to train mode
CustomBlock set to eval mode
CustomBlock set to eval mode
Output training mode: tensor([[0.8274, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000]], grad_fn=<MulBackward0>)
Output eval mode: tensor([[0.4137, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000]])


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:



## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

In [10]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


nn.Sequential output shape: torch.Size([5, 2])


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

In [11]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

nn.ModuleList output shape: torch.Size([5, 2])


## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

In [12]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)

nn.ModuleDict output shape: torch.Size([5, 2])


## Type hints and Shape Information

For production code, a practical pattern is to use `Tensor` for type hints and document the expected shape next to the signature.

`Tensor` tells us what kind of object is expected, while `[B, C, H, W]` documents the tensor dimensions explicitly.


In [13]:
from torch import Tensor

def normalize_images(x: Tensor) -> Tensor:
    # x: [B, C, H, W]
    return x / 255.0



## Работа на семинаре

### LSTM

![lstm](assets/LSTM.png)

### Inception

![inception](assets/inception.png)

### SE

![se](assets/SqueezeAndExcite.png)

### Selective Kernel

![selective](assets/SelectiveKernel.png)


### PatchMerger

![patchmerger](assets/PatchMerger.png)


## Домашнее задание

## Задание 1, реализуйте требуемые блоки

## ResNet Block (0.1 балл)

![Resnet](assets/ResBlock.png)

https://arxiv.org/pdf/1512.03385

In [14]:
class ResidualBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, C, H, W]
        return x + self.block(x)

## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](assets/DepthWiseConv.png)

https://arxiv.org/pdf/1610.02357

In [15]:
class SeparableConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int,
                 kernel_size: int = 3, stride: int = 1, bias: bool = False):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size,
                                   stride=stride, padding=kernel_size // 2,
                                   groups=in_channels, bias=bias)
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=bias)

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, C_in, H, W] -> [B, C_out, H', W']
        return self.pointwise(self.depthwise(x))

## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$



https://arxiv.org/abs/1409.0473


https://arxiv.org/abs/1508.04025

In [16]:
class VanillaAttention(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.W_align = nn.Linear(d_model, d_model, bias=False)
        self.W_value = nn.Linear(d_model, d_model, bias=False)
        self.W_query = nn.Linear(d_model, d_model, bias=False)
        self.softmax = nn.Softmax(dim=1)
        self.tanh = nn.Tanh()

    def forward(self, query: Tensor, key: Tensor) -> Tensor:
        # query: [B, d], key: [B, L, d] -> [B, d]
        q = self.W_align(query).unsqueeze(1)            
        score = q @ key.transpose(-2, -1)                
        att = self.softmax(score)                       
        context = (att @ key).squeeze(1)               
        return self.tanh(self.W_value(context) + self.W_query(query))   

## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$



https://arxiv.org/abs/1706.03762


In [17]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, query: Tensor, key: Tensor, value: Tensor):
        # query: [B, Lq, d_k], key: [B, Lk, d_k], value: [B, Lk, d_v] -> [B, Lq, d_v]
        d_k = query.size(-1)

        logits = query @ key.transpose(-2, -1) / (d_k ** 0.5)

        weights = torch.softmax(logits, dim=-1)

        return weights @ value

## Multihead Attention (0.1 балл)

![MultiheadAttention](assets/MultiheadAttention.webp)

https://arxiv.org/abs/1706.03762


In [18]:
def split_heads(x: torch.Tensor, n_heads: int) -> torch.Tensor:
    """(B, L, d_model) -> (B, n_heads, L, d_k)."""
    batch, seq_len, d_model = x.shape
    return x.view(batch, seq_len, n_heads, d_model // n_heads).transpose(1, 2)


def merge_heads(x: torch.Tensor) -> torch.Tensor:
    """(B, n_heads, L, d_k) -> (B, L, d_model)."""
    batch, n_heads, seq_len, d_k = x.shape
    return x.transpose(1, 2).reshape(batch, seq_len, n_heads * d_k)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()

        assert d_model % n_heads == 0

        self.n_heads = n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention()

    def forward(self, query, key, value):
        # query: [B, Lq, d_model], key/value: [B, Lk, d_model] -> [B, Lq, d_model]
        Q = split_heads(self.q_proj(query), self.n_heads)
        K = split_heads(self.k_proj(key), self.n_heads)
        V = split_heads(self.v_proj(value), self.n_heads)

        context = self.attention(Q, K, V)

        return self.out_proj(merge_heads(context))

## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](assets/TransformerEncoder.png)


https://arxiv.org/abs/1706.03762

In [19]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.mha = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, L, d_model] -> [B, L, d_model]
        h = self.ln1(x)                    
        x = x + self.mha(h, h, h)        
        x = x + self.mlp(self.ln2(x))       
        return x

## MLP Mixer (0.1 балл)


![MLPMixer](assets/MLPMixer.png)


https://arxiv.org/abs/2105.01601

In [20]:
class MLPMixerBlock(nn.Module):
    def __init__(self, patches: int, channels: int, patches_hidden: int, channels_hidden: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(channels)
        self.mlp1 = nn.Sequential(                     
            nn.Linear(patches, patches_hidden),
            nn.GELU(),
            nn.Linear(patches_hidden, patches),
        )
        self.ln2 = nn.LayerNorm(channels)
        self.mlp2 = nn.Sequential(                       
            nn.Linear(channels, channels_hidden),
            nn.GELU(),
            nn.Linear(channels_hidden, channels),
        )

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, P, C] -> [B, P, C]
        y = self.ln1(x).transpose(1, 2)             
        x = x + self.mlp1(y).transpose(1, 2)           
        x = x + self.mlp2(self.ln2(x))                
        return x

## ConvMixer (0.1 балл)

![ConvMixer](assets/ConvMixer.png)


https://arxiv.org/abs/2201.09792

In [21]:
class ConvMixer(nn.Module):
    def __init__(self, in_channels: int, kernel_size: int = 3):
        super().__init__()

        self.depthwise = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size, groups=in_channels, padding=kernel_size // 2),
            nn.ReLU(),
            nn.BatchNorm2d(in_channels),
        )
        self.pointwise = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 1),
            nn.ReLU(),
            nn.BatchNorm2d(in_channels),
        )

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, dim, H, W] -> [B, dim, H, W]
        x = x + self.depthwise(x)  
        return self.pointwise(x)     


## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

Ответ: 
1. MLPMixer и ConvMixer работают почти также эффективно как MHA, именно из-за устройства MultiHeadAttention, так как он тоже производит смешевание информации между патчами и между каналами. А MLPMixer и ConvMixer унаследовали skip-connections, нормализацию и большую глубину. Все три модели работают на смешивании информации, совмещая лучшие практики из MultiHeadAttention.

2. Взять вход $x : (B, P, C)$, функцию смешшивания $\text{Mix}(x)$, $\text{LayerNorm}(x)$ и слой проекции $\text{Proj}(x)$:

Затем общая формула для слоя $Layer(x) = \text{Proj}(\text{LayerNorm}(x + \text{Mix}(\text{LayerNorm}(x))))$.

3. 
| | ConvMixer | MLP-Mixer | MultiHead Attention |
|---|---|---|---|
| Как смешиваются патчи | локально (окно $k \times k$), веса фиксированные | глобально, веса фиксированные | глобально, веса зависят от входа |
| Сложность по числу патчей $S$ | линейная | линейная | квадратичная ($O(S^2)$) |
| Размер входа | любой | фиксированное число патчей | любая длина |